# VirtualDB Tutorial: SQL-First Cross-Dataset Queries

The `VirtualDB` class provides a SQL query interface across heterogeneous
Huggingface datasets stored in Parquet format. See the [VirtualDB configuration
documentation](../virtual_db_configuration.md) for more details on how to set up your
datasets for use with `VirtualDB`.

## Configuration

The configuration for `VirtualDB` is defined in a YAML file that specifies the datasets to include, their locations, and any relevant metadata or mappings. Below is an example configuration:

In [1]:
config_yaml = """
repositories:
  BrentLab/harbison_2004:
    tags:
      assay: binding
      method: chip-chip
      organism: yeast
    dataset:
      harbison_2004:
        db_name: harbison
        sample_id:
          field: sample_id
        carbon_source:
          field: condition
          path: media.carbon_source.compound
        temperature_celsius:
          field: condition
          path: temperature_celsius
          dtype: numeric
        regulator_locus_tag:
          field: regulator_locus_tag
        regulator_symbol:
          field: regulator_symbol

  BrentLab/kemmeren_2014:
    tags:
      assay: perturbation
      method: microarray
      organism: yeast
    dataset:
      kemmeren_2014:
        db_name: kemmeren
        sample_id:
          field: sample_id
        carbon_source:
          path: experimental_conditions.media.carbon_source.compound
        temperature_celsius:
          path: experimental_conditions.temperature_celsius
          dtype: numeric
        regulator_locus_tag:
          field: regulator_locus_tag
        regulator_symbol:
          field: regulator_symbol

  BrentLab/hackett_2020:
    # Repo-level tags apply to all datasets in this repository
    tags:
      method: test_overwrite
      organism: yeast
    dataset:
      hackett_2020:
        # Dataset-level tags: 'assay' is new,
        # 'method' overrides the repo-level value
        tags:
          assay: perturbation
          method: overexpression
        db_name: hackett
        sample_id:
          field: sample_id
        carbon_source:
          path: experimental_conditions.media.carbon_source.compound
        temperature_celsius:
          path: experimental_conditions.temperature_celsius
          dtype: numeric
        regulator_locus_tag:
          field: regulator_locus_tag
        regulator_symbol:
          field: regulator_symbol

  BrentLab/yeast_comparative_analysis:
    dataset:
      dto:
        dto_pvalue:
          field: dto_empirical_pvalue
        dto_fdr:
          field: dto_fdr
        links:
          binding_id:
            - [BrentLab/harbison_2004, harbison_2004]
          perturbation_id:
            - [BrentLab/kemmeren_2014, kemmeren_2014]
            - [BrentLab/hackett_2020, hackett_2020]

factor_aliases:
  carbon_source:
    glucose: [D-glucose, dextrose, glu]
    galactose: [D-galactose, gal]
    raffinose: [D-raffinose]

missing_value_labels:
  carbon_source: unspecified

description:
  carbon_source: The carbon source provided during growth
  temperature_celsius: Growth temperature in degrees Celsius
"""

import tempfile
from pathlib import Path

temp_config = Path(tempfile.mkdtemp()) / "vdb_config.yaml"
temp_config.write_text(config_yaml)
print(f"Config saved to: {temp_config}")

Config saved to: /tmp/tmp2k9hi3gb/vdb_config.yaml


## Initializing VirtualDB

Creating a VirtualDB instance loads and validates the config, downloads any
necessary data, and registers all views immediately.

In [2]:
from labretriever.virtual_db import VirtualDB

# Pass an HF token if the repos are private:
# vdb = VirtualDB(str(temp_config), token="hf_...")
vdb = VirtualDB(str(temp_config))
print(repr(vdb))

/home/chase/code/labretriever/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 36 files: 100%|██████████| 36/36 [00:00<00:00, 10365.55it/s]
Key 'carbon_source' not found at path 'media.carbon_source' (current keys: ['name'])
Key 'carbon_source' not found at path 'media.carbon_source' (current keys: ['name'])
Key 'carbon_source' not found at path 'media.carbon_source' (current keys: ['name'])
Key 'temperature_celsius' not found at path 'temperature_celsius' (current keys: ['description', 'growth_phase_at_harvest', 'media'])
Key 'temperature_celsius' not found at path 'temperature_celsius' (current keys: ['description', 'growth_phase_at_harvest', 'media', 'chemical_treatment'])
Key 'temperature_celsius' not found at path 'temperature_celsius' (current keys: ['description', 'growth_phase_at_h

VirtualDB(4 repos, 4 datasets, 7 views)


## Listing datasets

To list the datasets available in the VirtualDB instance, use `get_datasets()`.

In [3]:
print("\nDatasets:")
for dataset in vdb.get_datasets():
    print(f"- {dataset}")


Datasets:
- dto
- hackett
- harbison
- kemmeren


## Tags

Tags are arbitrary key/value annotations defined in the configuration. They
follow the same hierarchy as property mappings: repo-level tags apply to all
datasets in that repository, and dataset-level tags override repo-level tags
with the same key.

Use `config.get_tags(repo_id, config_name)` to retrieve the merged tags for
any dataset.

In [4]:

# Tags are accessible directly from the VirtualDB instance using the db_name.
# No need to import MetadataConfig or specify repo_id.
print("harbison tags:", vdb.get_tags("harbison"))
print("kemmeren tags:", vdb.get_tags("kemmeren"))

# Hackett has tags at both levels:
# 'organism' comes from the repo level only,
# 'assay' is added at the dataset level only,
# 'method' is defined at both levels -- the dataset value wins.
print("hackett tags:", vdb.get_tags("hackett"))

# Dataset with no tags returns empty dict
print("dto tags:", vdb.get_tags("dto"))

harbison tags: {'assay': 'binding', 'method': 'chip-chip', 'organism': 'yeast'}
kemmeren tags: {'assay': 'perturbation', 'method': 'microarray', 'organism': 'yeast'}
hackett tags: {'method': 'overexpression', 'organism': 'yeast', 'assay': 'perturbation'}
dto tags: {}


## Schema Discovery

Use `tables()`, `describe()`, `get_fields()`, and `get_common_fields()`
to explore the registered views before writing SQL.

Note that primary datasets get **two** views each:
- `<db_name>` -- the full measurement-level data (one row per sample-target pair)
- `<db_name>_meta` -- deduplicated sample-level metadata (one row per sample),
  including derived columns from config property mappings (e.g., `carbon_source` resolved from DataCard field definitions, with factor aliases applied).

In [5]:
# List all registered views
print("Registered views:")
for name in vdb.tables():
    print(f"  {name}")

Registered views:
  dto_expanded
  hackett
  hackett_meta
  harbison
  harbison_meta
  kemmeren
  kemmeren_meta


In [6]:
# The _meta view has sample-level metadata plus derived columns
# (carbon_source, temperature_celsius resolved from condition definitions)
vdb.describe("harbison_meta")

,table,column_name,column_type,null,key,default,extra
0,harbison_meta,sample_id,INTEGER,YES,None,None,None
1,harbison_meta,condition,VARCHAR,YES,None,None,None
2,harbison_meta,regulator_locus_tag,VARCHAR,YES,None,None,None
3,harbison_meta,regulator_symbol,VARCHAR,YES,None,None,None
4,harbison_meta,carbon_source,VARCHAR,YES,None,None,None
5,harbison_meta,temperature_celsius,DOUBLE,YES,None,None,None


In [7]:
# The full view has measurement-level data (one row per sample-target pair)
vdb.describe("harbison")

,table,column_name,column_type,null,key,default,extra
0,harbison,sample_id,INTEGER,YES,None,None,None
1,harbison,db_id,DOUBLE,YES,None,None,None
2,harbison,regulator_locus_tag,VARCHAR,YES,None,None,None
3,harbison,regulator_symbol,VARCHAR,YES,None,None,None
4,harbison,condition,VARCHAR,YES,None,None,None
5,harbison,target_locus_tag,VARCHAR,YES,None,None,None
6,harbison,target_symbol,VARCHAR,YES,None,None,None
7,harbison,effect,DOUBLE,YES,None,None,None
8,harbison,pvalue,DOUBLE,YES,None,None,None
9,harbison,carbon_source,VARCHAR,YES,None,None,None


In [8]:
# Columns common to ALL primary dataset views
print("Common fields:", vdb.get_common_fields())

Common fields: ['carbon_source', 'regulator_locus_tag', 'regulator_symbol', 'sample_id', 'temperature_celsius']


## Linked Property Columns

Some datasets expose multiple derived property columns in their  view
(for example  and ) that are both extracted
from the **same** per-sample experimental-condition field via field+path
property mappings.  Because these columns are co-derived, the values that can
appear in one column are constrained by what is selected in another: picking
 limits which  values are
meaningful.

 identifies such groups and enriches them
with per-level descriptions from the DataCard, making it straightforward to
build conditional filter interfaces or to understand which property columns
should be filtered together.

The method returns  for datasets that have no linked property groups
(fewer than two property columns share the same source field).

In [9]:
# Identify datasets that have hierarchically linked property columns.
# Two or more property columns are "linked" when they are both derived
# from the same per-sample experimental-condition field via field+path
# PropertyMappings in the VirtualDB configuration.
for dataset in vdb.get_datasets():
    info = vdb.get_condition_field_info(dataset)
    if info is None:
        continue
    for src_field, group in info.items():
        print(f"{dataset}: '{src_field}' links {group['property_cols']}")
        for level, desc in list(group["level_descriptions"].items())[:3]:
            print(f"  {level}: {desc}")

hackett: 'time' links ['carbon_source', 'temperature_celsius']
kemmeren: 'variable_in_wt' links ['carbon_source', 'temperature_celsius']


/tmp/ipykernel_69931/1092681254.py:6: DeprecationWarning: get_condition_field_info is deprecated; use get_column_metadata instead.
  info = vdb.get_condition_field_info(dataset)


## Missing Value Labels

When a property key is listed under `missing_value_labels`, every dataset
that does **not** have an explicit mapping for that property will still expose
the column in its `_meta` view, filled with the configured fallback string.

In the config above, `carbon_source: unspecified` is set in `missing_value_labels`.
All three datasets (harbison, kemmeren, hackett) happen to have an explicit
`carbon_source` mapping, so they resolve real values.

To demonstrate the fallback, we build a minimal config that omits `carbon_source`
from kemmeren. Without `missing_value_labels`, kemmeren would have no
`carbon_source` column at all. With it, the column appears with the default value.

In [10]:
minimal_yaml = """
repositories:
  BrentLab/harbison_2004:
    dataset:
      harbison_2004:
        db_name: harbison2
        sample_id:
          field: sample_id
        # harbison has carbon_source mapped via field+path
        carbon_source:
          field: condition
          path: media.carbon_source.compound

  BrentLab/kemmeren_2014:
    dataset:
      kemmeren_2014:
        db_name: kemmeren2
        sample_id:
          field: sample_id
        # kemmeren has NO carbon_source mapping -- fallback will apply

factor_aliases:
  carbon_source:
    glucose: [D-glucose, dextrose, glu]

missing_value_labels:
  carbon_source: unspecified
"""

import tempfile
from pathlib import Path
from labretriever.virtual_db import VirtualDB

p = Path(tempfile.mkdtemp()) / "minimal.yaml"
p.write_text(minimal_yaml)
vdb2 = VirtualDB(str(p))

# harbison resolves real values from DataCard definitions
print("harbison2 carbon_source values:")
print(vdb2.query("SELECT DISTINCT carbon_source FROM harbison2_meta"))

# kemmeren has no mapping -- gets the missing_value_labels fallback
print("\nkemmeren2 carbon_source values:")
print(vdb2.query("SELECT DISTINCT carbon_source FROM kemmeren2_meta"))

# Both views expose the column, enabling cross-dataset queries without COALESCE
print("\ncross-dataset query using carbon_source on both:")
print(vdb2.query("""
    SELECT 'harbison' AS dataset, carbon_source, COUNT(*) AS n
    FROM harbison2_meta GROUP BY carbon_source
    UNION ALL
    SELECT 'kemmeren' AS dataset, carbon_source, COUNT(*) AS n
    FROM kemmeren2_meta GROUP BY carbon_source
    ORDER BY dataset, carbon_source
"""))

p.unlink(missing_ok=True)


Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00, 7219.11it/s]
Key 'carbon_source' not found at path 'media.carbon_source' (current keys: ['name'])
Key 'carbon_source' not found at path 'media.carbon_source' (current keys: ['name'])
Key 'carbon_source' not found at path 'media.carbon_source' (current keys: ['name'])


harbison2 carbon_source values:
  carbon_source
0   D-galactose
1   D-raffinose
2       glucose
3   unspecified

kemmeren2 carbon_source values:
  carbon_source
0   unspecified

cross-dataset query using carbon_source on both:
    dataset carbon_source     n
0  harbison   D-galactose     4
1  harbison   D-raffinose     1
2  harbison       glucose   310
3  harbison   unspecified    37
4  kemmeren   unspecified  1484


## Querying VirtualDB

The `.query()` method executes SQL queries against the registered views. You can write complex SQL queries that join across multiple datasets, filter based on metadata, and aggregate results as needed.  

You can also use parameterized queries to safely inject variables into your SQL statements, and prepared statements for repeated queries with different parameters. 
Named prepared statements can be passed to `.prepare()` and then executed with
`.query()` with any parameterized values passed in as an arbitrary number of key/value
arguments.

In [11]:
# Query the _meta view for sample-level metadata (one row per sample)
# Note: carbon_source is derived from the condition column's DataCard definitions
# with factor aliases already applied (D-glucose -> glucose)
df_meta = vdb.query("SELECT * FROM harbison_meta LIMIT 5")
df_meta

,sample_id,condition,regulator_locus_tag,regulator_symbol,carbon_source,temperature_celsius
0,136,YPD,YGR040W,KSS1,glucose,37.0
1,94,YPD,YER130C,COM2,glucose,37.0
2,293,H2O2Lo,YOL028C,YAP7,glucose,37.0
3,325,YPD,YPL075W,GCR1,glucose,37.0
4,80,YPD,YDR520C,URC2,glucose,37.0


## 5. Parameterized Queries

Pass keyword arguments to `query()` and reference them with
DuckDB's `$name` syntax.

In [12]:
# A parameterized query has the following form, where `$reg` is a placeholder
# that gets replaced with the value provided in the `reg` argument.
vdb.query(
    "SELECT * FROM harbison WHERE regulator_symbol = $reg LIMIT 5",
    reg="REB1",
)

# A parameterized query can be saved for future use with the `.prepare()` method

,sample_id,db_id,regulator_locus_tag,regulator_symbol,condition,target_locus_tag,target_symbol,effect,pvalue,carbon_source,temperature_celsius
0,15,14.0,YBR049C,REB1,YPD,YPR204W,YPR204W,0.852889,0.769430,glucose,37.0
1,15,14.0,YBR049C,REB1,YPD,YPR203W,YPR203W,1.249003,0.112376,glucose,37.0
2,15,14.0,YBR049C,REB1,YPD,YPR202W,YPR202W,1.249003,0.112376,glucose,37.0
3,15,14.0,YBR049C,REB1,YPD,YPR201W,ARR3,1.513707,0.168133,glucose,37.0
4,15,14.0,YBR049C,REB1,YPD,YPR200C,ARR2,1.513707,0.168133,glucose,37.0


##  Prepared Queries

Use `prepare()` to register a named, reusable query template.
Then call it by name via `query()`.

In [13]:
# Register a prepared query
vdb.prepare("glucose_regs", """
    SELECT regulator_symbol, COUNT(*) AS n
    FROM harbison_meta
    WHERE carbon_source = $cs
    GROUP BY regulator_symbol
    HAVING n >= $min_n
    ORDER BY n DESC
""")

# note that rather than a SQL statement, we pass in the name of the prepared query
# and provide the appropriate parameters
vdb.query("glucose_regs", cs="glucose", min_n=2)

,regulator_symbol,n
0,MSN2,6
1,MSN4,5
2,DIG1,4
3,HSF1,4
4,STE12,4
...,...,...
58,HAP4,2
59,YAP3,2
60,GLN3,2
61,DAL81,2


## 7. Comparative Dataset Views

Comparative datasets (those with `links`) get an extra view type:

**`<name>_expanded`**: For each composite ID field, adds two parsed columns:
- `<link_field>_source` -- the source dataset, aliased to `db_name` when
  the `repo_id;config_name` pair is in the VirtualDB config.
- `<link_field>_id` -- the sample_id component.

This makes it easy to join or filter by source dataset without manually
parsing composite IDs.

In [14]:
# The expanded view has parsed _source and _id columns for each link field
vdb.query("SELECT * FROM dto_expanded LIMIT 3")

,binding_id,perturbation_id,binding_rank_threshold,perturbation_rank_threshold,binding_set_size,perturbation_set_size,dto_fdr,dto_empirical_pvalue,pr_ranking_column,binding_repo_dataset,perturbation_repo_dataset,binding_id_id,binding_id_source,perturbation_id_id,perturbation_id_source
0,BrentLab/harbison_2004;harbison_2004;105,BrentLab/hughes_2006;overexpression;10,11.0,206.0,12.0,206.0,0.041293,0.017,log2fc,harbison_2004-harbison_2004,hughes_2006-overexpression,105,harbison,10,BrentLab/hughes_2006;overexpression
1,BrentLab/harbison_2004;harbison_2004;108,BrentLab/hughes_2006;overexpression;11,60.0,67.0,60.0,67.0,0.054284,0.000,log2fc,harbison_2004-harbison_2004,hughes_2006-overexpression,108,harbison,11,BrentLab/hughes_2006;overexpression
2,BrentLab/harbison_2004;harbison_2004;109,BrentLab/hughes_2006;overexpression;11,27.0,1265.0,27.0,1265.0,0.123214,0.057,log2fc,harbison_2004-harbison_2004,hughes_2006-overexpression,109,harbison,11,BrentLab/hughes_2006;overexpression


In [15]:
# Join harbison metadata to dto via the expanded view's parsed columns
vdb.query("""
    SELECT h.*, d.dto_empirical_pvalue, d.dto_fdr
    FROM harbison_meta h
    JOIN dto_expanded d
        ON CAST(h.sample_id AS VARCHAR) = d.binding_id_id
        AND d.binding_id_source = 'harbison'
    WHERE d.dto_empirical_pvalue <= 0.01
    ORDER BY d.dto_empirical_pvalue
    LIMIT 10
""")

,sample_id,condition,regulator_locus_tag,regulator_symbol,carbon_source,temperature_celsius,dto_empirical_pvalue,dto_fdr
0,7,H2O2Hi,YBL103C,RTG3,glucose,37.0,0.0,0.260226
1,317,YPD,YOR372C,NDD1,glucose,37.0,0.0,0.015880
2,273,H2O2Hi,YNL068C,FKH2,glucose,37.0,0.0,0.139295
3,17,BUT14,YBR083W,TEC1,glucose,37.0,0.0,0.094389
4,317,YPD,YOR372C,NDD1,glucose,37.0,0.0,0.024633
5,261,Alpha,YMR043W,MCM1,glucose,37.0,0.0,0.018293
6,308,YPD,YOR140W,SFL1,glucose,37.0,0.0,0.027873
7,235,SM,YLR451W,LEU3,unspecified,37.0,0.0,0.088086
8,301,H2O2Hi,YOR028C,CIN5,glucose,37.0,0.0,0.100408
9,261,Alpha,YMR043W,MCM1,glucose,37.0,0.0,0.054081


In [16]:
# Cross-dataset join: harbison binding with hackett perturbation data
# via the DTO comparative dataset
vdb.query("""
    SELECT
        h.sample_id        AS harbison_sample_id,
        h.regulator_symbol,
        d.dto_empirical_pvalue,
        d.perturbation_id_id AS hackett_sample_id
    FROM harbison_meta h
    JOIN dto_expanded d
        ON CAST(h.sample_id AS VARCHAR) = d.binding_id_id
        AND d.binding_id_source = 'harbison'
    WHERE d.dto_empirical_pvalue <= 0.01
    ORDER BY d.dto_empirical_pvalue
    LIMIT 10
""")

,harbison_sample_id,regulator_symbol,dto_empirical_pvalue,hackett_sample_id
0,118,HSF1,0.0,88
1,15,REB1,0.0,100_242
2,31,RPN4,0.0,229
3,162,XBP1,0.0,741
4,303,CIN5,0.0,1369
5,314,HAP5,0.0,1488
6,36,MBP1,0.0,212
7,240,YAP1,0.0,1092
8,330,CUP9,0.0,1551
9,9,RTG3,0.0,48


## A realistic example

Hackett has multiple experimental conditions that are unique to that dataset. There are
some regulators which have replicates within those conditions. We need to find those 
regulators and design a query which returns only 1 sample per condition set.

In [17]:
# Query hackett to find regulators with multiple samples in the same (time, mechanism)
# condition
vdb.query("""
    SELECT regulator_symbol, time, mechanism, restriction, COUNT(*) AS n
    FROM hackett_meta
    GROUP BY regulator_symbol, time, mechanism, restriction
    HAVING n > 1
    ORDER BY n DESC
""")

,regulator_symbol,time,mechanism,restriction,n
0,SWI1,10.0,ZEV,P,3
1,SWI1,90.0,ZEV,P,3
2,SWI1,20.0,ZEV,P,3
3,SWI1,45.0,ZEV,P,3
4,SWI1,15.0,ZEV,P,3
5,SWI1,30.0,ZEV,P,3
6,SWI1,5.0,ZEV,P,3
7,SWI1,0.0,ZEV,P,3
8,RDS2,30.0,ZEV,P,2
9,RDS2,10.0,ZEV,P,2


In [18]:
# SWI1 has 3 samples at time=20, mechanism=ZEV. Let's look at just those samples
vdb.query("""
    SELECT *
    FROM hackett_meta
    WHERE regulator_symbol = 'SWI1'
      AND time = 20
      AND mechanism = 'ZEV'
""")

,sample_id,date,mechanism,regulator_locus_tag,regulator_symbol,restriction,strain,time,carbon_source,temperature_celsius
0,1636,20161117,ZEV,YPL016W,SWI1,P,SMY2266c,20.0,glucose,30.0
1,1620,20161117,ZEV,YPL016W,SWI1,P,SMY2266a,20.0,glucose,30.0
2,1628,20161117,ZEV,YPL016W,SWI1,P,SMY2266b,20.0,glucose,30.0


In [19]:
# In this case, there are three strains with otherwise the same experimental conditions.
# Rather than trying to choose among these right now, we might just want to get a
# unique list of the regulators with replicates in order to exclude them from an
# analysis that doesn't expect replicates.
replicated_hackett_regulators = vdb.query("""
    SELECT DISTINCT regulator_symbol
    FROM hackett_meta
    GROUP BY regulator_symbol, time, mechanism, restriction
    HAVING COUNT(*) > 1
""").regulator_symbol.tolist()
print(replicated_hackett_regulators)

['GCN4', 'RDS2', 'MAC1', 'SWI1']


In [20]:
# GEV is another "regulator" we want to exclude
replicated_hackett_regulators.append("GEV")
print(replicated_hackett_regulators)

['GCN4', 'RDS2', 'MAC1', 'SWI1', 'GEV']


In [21]:
vdb.query("SELECT * FROM dto_expanded")

,binding_id,perturbation_id,binding_rank_threshold,perturbation_rank_threshold,binding_set_size,perturbation_set_size,dto_fdr,dto_empirical_pvalue,pr_ranking_column,binding_repo_dataset,perturbation_repo_dataset,binding_id_id,binding_id_source,perturbation_id_id,perturbation_id_source
0,BrentLab/harbison_2004;harbison_2004;105,BrentLab/hughes_2006;overexpression;10,11.0,206.0,12.0,206.0,0.041293,0.017,log2fc,harbison_2004-harbison_2004,hughes_2006-overexpression,105,harbison,10,BrentLab/hughes_2006;overexpression
1,BrentLab/harbison_2004;harbison_2004;108,BrentLab/hughes_2006;overexpression;11,60.0,67.0,60.0,67.0,0.054284,0.000,log2fc,harbison_2004-harbison_2004,hughes_2006-overexpression,108,harbison,11,BrentLab/hughes_2006;overexpression
2,BrentLab/harbison_2004;harbison_2004;109,BrentLab/hughes_2006;overexpression;11,27.0,1265.0,27.0,1265.0,0.123214,0.057,log2fc,harbison_2004-harbison_2004,hughes_2006-overexpression,109,harbison,11,BrentLab/hughes_2006;overexpression
3,BrentLab/harbison_2004;harbison_2004;112,BrentLab/hughes_2006;overexpression;12,532.0,1093.0,532.0,1093.0,0.436305,0.092,log2fc,harbison_2004-harbison_2004,hughes_2006-overexpression,112,harbison,12,BrentLab/hughes_2006;overexpression
4,BrentLab/harbison_2004;harbison_2004;113,BrentLab/hughes_2006;overexpression;12,10.0,556.0,10.0,556.0,0.017567,0.002,log2fc,harbison_2004-harbison_2004,hughes_2006-overexpression,113,harbison,12,BrentLab/hughes_2006;overexpression
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32165,BrentLab/callingcards;2026_analysis_set;45,BrentLab/kemmeren_2014;kemmeren_2014;1239,1.0,2.0,1.0,2.0,0.000000,0.011,pvalue,callingcards-2026_analysis_set,kemmeren_2014-kemmeren_2014,45,BrentLab/callingcards;2026_analysis_set,1239,kemmeren
32166,BrentLab/callingcards;2026_analysis_set;37,BrentLab/kemmeren_2014;kemmeren_2014;683,14.0,14.0,14.0,14.0,0.001267,0.000,pvalue,callingcards-2026_analysis_set,kemmeren_2014-kemmeren_2014,37,BrentLab/callingcards;2026_analysis_set,683,kemmeren
32167,BrentLab/callingcards;2026_analysis_set;34,BrentLab/kemmeren_2014;kemmeren_2014;312,550.0,200.0,552.0,200.0,0.408162,0.101,pvalue,callingcards-2026_analysis_set,kemmeren_2014-kemmeren_2014,34,BrentLab/callingcards;2026_analysis_set,312,kemmeren
32168,BrentLab/callingcards;2026_analysis_set;20,BrentLab/kemmeren_2014;kemmeren_2014;88,10.0,72.0,10.0,75.0,0.002128,0.000,pvalue,callingcards-2026_analysis_set,kemmeren_2014-kemmeren_2014,20,BrentLab/callingcards;2026_analysis_set,88,kemmeren


In [22]:
# We can remove those regulators from our query using a parameterized query
hackett_harbison_dto = vdb.query("""
SELECT h.sample_id, h.regulator_symbol, h.time, h.mechanism,
       dto.*
FROM hackett_meta h
LEFT JOIN (
    SELECT *
    FROM dto_expanded
) AS dto
ON CAST(h.sample_id AS VARCHAR) = dto.perturbation_id_id
WHERE h.regulator_symbol NOT IN $replicated_hacket_regulators
    AND h.mechanism = 'ZEV'
    AND h.restriction = 'P'
    AND h.time = 15
ORDER BY h.regulator_symbol, h.time, h.mechanism
""",
    replicated_hacket_regulators=replicated_hackett_regulators
)
print(hackett_harbison_dto.head())

   sample_id regulator_symbol  time mechanism  \
0        448             ACA1  15.0       ZEV   
1        448             ACA1  15.0       ZEV   
2        448             ACA1  15.0       ZEV   
3        448             ACA1  15.0       ZEV   
4        448             ACA1  15.0       ZEV   

                                     binding_id  \
0  BrentLab/callingcards;annotated_features;189   
1       BrentLab/harbison_2004;harbison_2004;88   
2  BrentLab/callingcards;annotated_features;156   
3  BrentLab/callingcards;annotated_features;126   
4  BrentLab/callingcards;annotated_features;146   

                          perturbation_id  binding_rank_threshold  \
0  BrentLab/hackett_2020;hackett_2020;448                   194.0   
1  BrentLab/hackett_2020;hackett_2020;448                   122.0   
2  BrentLab/hackett_2020;hackett_2020;448                   374.0   
3  BrentLab/hackett_2020;hackett_2020;448                   437.0   
4  BrentLab/hackett_2020;hackett_2020;448            

## Citations

Citation metadata is defined in the HuggingFace dataset card YAML, not in the VirtualDB configuration file. `get_citation()` reads from the fetched DataCard, so it returns a value only when the upstream dataset card includes a `citation:` field.

Citations follow the same override hierarchy as experimental conditions: a dataset-level citation (on a specific config) overrides a repository-level citation. `get_citation()` handles the fallback automatically.

If the dataset card has no citation defined at either level, `get_citation()` returns `None`.

In [23]:
# Retrieve the citation for each dataset.
# Returns None when no citation field is present in the upstream DataCard.
for ds in vdb.get_datasets():
    citation = vdb.get_citation(ds)
    print(f"{ds}: {citation}")

dto: None
hackett: https://doi.org/10.15252/msb.20199174
harbison: https://doi.org/10.1038/nature02800
kemmeren: https://doi.org/10.1016/j.cell.2014.02.054


In [24]:
# Clean up temp file
temp_config.unlink(missing_ok=True)